# Levee Detection — D-LinkNet34 Semantic Segmentation

Alternative deep learning approach to the XGBoost pixel classifier.
Uses a ResNet34 encoder with dilated central block (D-Block) and clDice loss
to preserve linear connectivity of levee predictions.

**Pipeline:**
1. Architecture — D-LinkNet34 with corrected filter dimensions
2. clDice loss — soft skeleton aware loss function
3. Dataset — patch-based sampling from feature stack + label mask
4. Training loop — with early stopping and checkpointing
5. Tiled inference — full scene probability map
6. Postprocessing — binary mask → skeleton → GeoPackage vector

**Input (shared with levee_detection.ipynb):**
- `feature_stack.tif` — 18-band GeoTIFF, EPSG:2180, 10m
- `label_mask.tif` — binary raster, 1=levee, 0=non-levee

**Install:**
```
pip install torch torchvision numpy geopandas pyogrio scikit-image tqdm matplotlib
```

---
## Section 0 — Configuration

In [ ]:
from pathlib import Path
import torch

# ── Shared inputs from levee_detection.ipynb ───────────────────────────────
FEATURE_STACK_PATH = Path(r'C:/data/model/feature_stack.tif')
LABEL_MASK_PATH    = Path(r'C:/data/model/label_mask.tif')

# ── Output directory ───────────────────────────────────────────────────────
OUT_DIR = Path(r'C:/data/model/dlinknet')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model parameters ───────────────────────────────────────────────────────
NUM_CHANNELS  = 18       # all feature stack bands
PATCH_SIZE    = 256      # training patch size (pixels)
TILE_SIZE     = 512      # inference tile size
TILE_OVERLAP  = 64       # overlap between tiles to avoid edge artefacts
BATCH_SIZE    = 8
N_EPOCHS      = 50
LR            = 1e-4
PATIENCE      = 10       # early stopping patience (epochs)
N_PATCHES     = 4000     # patches sampled for training
POS_RATIO     = 0.5      # fraction of patches centred on levee pixels
RANDOM_STATE  = 42

# ── clDice loss parameters ─────────────────────────────────────────────────
CLDICE_ALPHA  = 0.5      # 0 = pure Dice, 1 = pure clDice
CLDICE_ITER   = 10       # soft skeleton iterations

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Section 1 — Raster I/O (GDAL)

In [ ]:
import numpy as np
from osgeo import gdal, ogr

gdal.UseExceptions()


def read_multiband(path: Path) -> tuple:
    """
    Reads all bands from a multiband GeoTIFF.
    Returns (array float32 shape (C, H, W), geo_info dict).
    NoData replaced with 0.0.
    """
    ds = gdal.Open(str(path), gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f'Cannot open: {path}')

    arrays = []
    for b in range(1, ds.RasterCount + 1):
        band   = ds.GetRasterBand(b)
        arr    = band.ReadAsArray().astype(np.float32)
        nodata = band.GetNoDataValue()
        if nodata is not None:
            arr[arr == nodata] = 0.0
        arr = np.nan_to_num(arr, nan=0.0)
        arrays.append(arr)

    geo_info = {
        'geotransform': ds.GetGeoTransform(),
        'projection':   ds.GetProjection(),
        'nrows':        ds.RasterYSize,
        'ncols':        ds.RasterXSize,
    }
    ds = None
    return np.stack(arrays, axis=0), geo_info


def read_band(path: Path) -> tuple:
    """Reads single-band GeoTIFF. Returns (array float32 H x W, geo_info)."""
    ds   = gdal.Open(str(path), gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f'Cannot open: {path}')
    band = ds.GetRasterBand(1)
    arr  = band.ReadAsArray().astype(np.float32)
    nodata = band.GetNoDataValue()
    if nodata is not None:
        arr[arr == nodata] = 0.0
    geo_info = {
        'geotransform': ds.GetGeoTransform(),
        'projection':   ds.GetProjection(),
        'nrows':        ds.RasterYSize,
        'ncols':        ds.RasterXSize,
    }
    ds = None
    return arr, geo_info


def write_singleband_tiff(path: Path, arr: np.ndarray, geo_info: dict, nodata: float = -9999.0):
    """Writes a single 2D float32 array as GeoTIFF."""
    nrows, ncols = arr.shape
    driver = gdal.GetDriverByName('GTiff')
    ds = driver.Create(
        str(path), ncols, nrows, 1, gdal.GDT_Float32,
        options=['COMPRESS=LZW', 'BIGTIFF=IF_SAFER']
    )
    ds.SetGeoTransform(geo_info['geotransform'])
    ds.SetProjection(geo_info['projection'])
    out = np.where(np.isnan(arr), nodata, arr).astype(np.float32)
    ds.GetRasterBand(1).WriteArray(out)
    ds.GetRasterBand(1).SetNoDataValue(nodata)
    ds.FlushCache()
    ds = None
    print(f'  Saved: {path.name}')


print('Loading feature stack and label mask...')
features, geo_ref = read_multiband(FEATURE_STACK_PATH)   # (18, H, W)
labels,   _       = read_band(LABEL_MASK_PATH)           # (H, W)

print(f'  Feature stack : {features.shape}  dtype={features.dtype}')
print(f'  Label mask    : {labels.shape}    dtype={labels.dtype}')
print(f'  Positive px   : {int(labels.sum()):,} / {labels.size:,}')


---
## Section 2 — D-LinkNet34 Architecture

Fixes applied over the original code provided:
- `filters` list defined correctly as `[64, 128, 256, 512]`
- `filters[0]` used consistently in decoder and final layers
- `soft_erode`/`soft_dilate` guard removed — input always 4D
- Weight initialisation for arbitrary number of input channels
- `pretrained=True` replaced with current torchvision API

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights


# ── Soft morphological operators ──────────────────────────────────────────

def soft_erode(img: torch.Tensor) -> torch.Tensor:
    """
    Differentiable approximation of morphological erosion via min-pooling.
    Applied separately in horizontal and vertical direction.
    Input shape: (B, C, H, W).
    """
    p1 = -F.max_pool2d(-img, (3, 1), (1, 1), (1, 0))
    p2 = -F.max_pool2d(-img, (1, 3), (1, 1), (0, 1))
    return torch.min(p1, p2)


def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    """Differentiable approximation of morphological dilation via max-pooling."""
    return F.max_pool2d(img, (3, 3), (1, 1), (1, 1))


def soft_open(img: torch.Tensor) -> torch.Tensor:
    """Morphological opening: erosion followed by dilation."""
    return soft_dilate(soft_erode(img))


def soft_skeletonize(img: torch.Tensor, iters: int = 10) -> torch.Tensor:
    """
    Iterative soft skeletonization via ReLU residuals.
    Progressively removes outer pixels down to the centreline.
    iters=10 balances accuracy and training speed.
    """
    img1 = soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(iters):
        img  = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel  = skel + F.relu(delta - skel * delta)
    return skel


# ── clDice loss ───────────────────────────────────────────────────────────

class SoftClDiceLoss(nn.Module):
    """
    Combined Dice + clDice loss.
    clDice penalises topological disconnections along the levee centreline.
    alpha=0 → pure Dice, alpha=1 → pure clDice.
    """
    def __init__(self, iters: int = 10, smooth: float = 1.0, alpha: float = 0.5):
        super().__init__()
        self.iters  = iters
        self.smooth = smooth
        self.alpha  = alpha

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        skel_pred = soft_skeletonize(y_pred, self.iters)
        skel_true = soft_skeletonize(y_true, self.iters)

        # Topology precision and sensitivity
        tprec = (torch.sum(skel_pred * y_true) + self.smooth) / \
                (torch.sum(skel_pred) + self.smooth)
        tsens = (torch.sum(skel_true * y_pred) + self.smooth) / \
                (torch.sum(skel_true) + self.smooth)
        cl_dice_loss = 1.0 - 2.0 * tprec * tsens / (tprec + tsens + 1e-8)

        # Standard volumetric Dice
        intersection = torch.sum(y_pred * y_true)
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / \
                    (torch.sum(y_pred) + torch.sum(y_true) + self.smooth)

        return (1.0 - self.alpha) * dice_loss + self.alpha * cl_dice_loss


# ── D-LinkNet34 blocks ────────────────────────────────────────────────────

class DBlock(nn.Module):
    """
    Dilated convolution cascade (rates 1, 2, 4, 8).
    Expands receptive field without losing spatial resolution.
    Outputs are additively fused to preserve all scales.
    """
    def __init__(self, channels: int):
        super().__init__()
        self.d1 = nn.Conv2d(channels, channels, 3, dilation=1, padding=1)
        self.d2 = nn.Conv2d(channels, channels, 3, dilation=2, padding=2)
        self.d3 = nn.Conv2d(channels, channels, 3, dilation=4, padding=4)
        self.d4 = nn.Conv2d(channels, channels, 3, dilation=8, padding=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        d1 = self.relu(self.d1(x))
        d2 = self.relu(self.d2(d1))
        d3 = self.relu(self.d3(d2))
        d4 = self.relu(self.d4(d3))
        return x + d1 + d2 + d3 + d4


class DecoderBlock(nn.Module):
    """
    LinkNet-style decoder: 1x1 bottleneck → transposed conv upsampling → 1x1 projection.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        mid = in_ch // 4
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(mid, mid, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, out_ch, 1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DLinkNet34(nn.Module):
    """
    D-LinkNet34 for binary levee segmentation.

    Fixes over original:
      - filters = [64, 128, 256, 512] explicitly defined
      - filters[0] used consistently in decoder tail and final layers
      - soft_erode/dilate always receive 4D tensors (guard removed)
      - Weight init: RGB pretrained weights averaged and tiled to num_channels
      - pretrained=True replaced with ResNet34_Weights.DEFAULT
    """
    def __init__(self, num_channels: int = 18, num_classes: int = 1):
        super().__init__()

        filters = [64, 128, 256, 512]  # FIX: was undefined in original

        resnet = resnet34(weights=ResNet34_Weights.DEFAULT)

        # Input conv adapted for num_channels SAR bands
        self.firstconv = nn.Conv2d(
            num_channels, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        # FIX: average pretrained RGB weights and tile to num_channels
        # Better than random init — low-level edge detectors transfer across modalities
        with torch.no_grad():
            mean_w = resnet.conv1.weight.data.mean(dim=1, keepdim=True)  # (64, 1, 7, 7)
            self.firstconv.weight.data = mean_w.repeat(1, num_channels, 1, 1)

        self.firstbn      = resnet.bn1
        self.firstrelu    = resnet.relu
        self.firstmaxpool = resnet.maxpool

        # ResNet34 encoder stages
        self.encoder1 = resnet.layer1   # 64 ch
        self.encoder2 = resnet.layer2   # 128 ch
        self.encoder3 = resnet.layer3   # 256 ch
        self.encoder4 = resnet.layer4   # 512 ch

        # Dilated centre block
        self.dblock = DBlock(filters[3])  # 512 ch

        # Decoder — FIX: filters[0] (64) used in tail, not bare 'filters'
        self.decoder4 = DecoderBlock(filters[3], filters[2])  # 512 → 256
        self.decoder3 = DecoderBlock(filters[2], filters[1])  # 256 → 128
        self.decoder2 = DecoderBlock(filters[1], filters[0])  # 128 → 64
        self.decoder1 = DecoderBlock(filters[0], filters[0])  # 64  → 64

        self.final = nn.Sequential(
            nn.ConvTranspose2d(filters[0], 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = self.firstconv(x)
        x  = self.firstbn(x)
        x  = self.firstrelu(x)
        x  = self.firstmaxpool(x)

        e1 = self.encoder1(x)    # 64 ch
        e2 = self.encoder2(e1)   # 128 ch
        e3 = self.encoder3(e2)   # 256 ch
        e4 = self.encoder4(e3)   # 512 ch

        e4 = self.dblock(e4)     # dilated context

        # Decoder with additive skip connections
        d4 = self.decoder4(e4) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)

        return torch.sigmoid(self.final(d1))  # (B, 1, H, W)


# Sanity check
model = DLinkNet34(num_channels=NUM_CHANNELS).to(DEVICE)
dummy = torch.zeros(2, NUM_CHANNELS, PATCH_SIZE, PATCH_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Architecture OK — output shape: {out.shape}')  # (2, 1, 256, 256)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params / 1e6:.1f} M')


---
## Section 3 — Patch Dataset

Patches sampled from the feature stack with stratified centring:
- `POS_RATIO` fraction of patches centred on a levee pixel
- Remaining patches sampled randomly (background)
- Per-channel normalisation (mean/std from training patches)

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


assert PATCH_SIZE % 2 == 0, f'PATCH_SIZE must be even, got {PATCH_SIZE}'


class LeveeDataset(Dataset):
    """
    Patch-based dataset for D-LinkNet34 training.

    Patches of shape (NUM_CHANNELS, PATCH_SIZE, PATCH_SIZE) are
    extracted at sampling time. Labels are binary masks (1, P, P).

    Stratified sampling ensures POS_RATIO of patches contain levee pixels.
    """
    def __init__(
        self,
        features: np.ndarray,       # (C, H, W)
        labels: np.ndarray,         # (H, W)
        patch_size: int,
        n_patches: int,
        pos_ratio: float = POS_RATIO,
        random_state: int = RANDOM_STATE,
        mean: np.ndarray = None,    # per-channel mean for normalisation
        std: np.ndarray = None,
    ):
        super().__init__()
        self.features   = features
        self.labels     = labels
        self.patch_size = patch_size
        self.mean       = mean
        self.std        = std
        self.coords     = self._sample_coords(n_patches, pos_ratio, random_state)

    def set_norm(self, mean: np.ndarray, std: np.ndarray):
        """Set normalisation statistics (call after computing from train split)."""
        self.mean = mean
        self.std  = std

    def _sample_coords(self, n_patches, pos_ratio, seed):
        rng = np.random.default_rng(seed)
        C, H, W = self.features.shape
        half    = self.patch_size // 2

        # Valid pixel range (avoid border)
        r_min, r_max = half, H - half - 1
        c_min, c_max = half, W - half - 1

        pos_yx = np.argwhere(
            (self.labels[r_min:r_max, c_min:c_max] == 1)
        ) + np.array([r_min, c_min])

        n_pos = int(n_patches * pos_ratio)
        n_neg = n_patches - n_pos

        # Positive centres
        if len(pos_yx) >= n_pos:
            pos_sampled = pos_yx[rng.choice(len(pos_yx), n_pos, replace=False)]
        else:
            pos_sampled = pos_yx[rng.choice(len(pos_yx), n_pos, replace=True)]

        # Negative centres (random)
        neg_r = rng.integers(r_min, r_max, n_neg)
        neg_c = rng.integers(c_min, c_max, n_neg)
        neg_sampled = np.stack([neg_r, neg_c], axis=1)

        coords = np.concatenate([pos_sampled, neg_sampled], axis=0)
        rng.shuffle(coords)
        return coords

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        r, c  = self.coords[idx]
        half  = self.patch_size // 2

        feat  = self.features[:, r - half:r + half, c - half:c + half].copy()
        label = self.labels[r - half:r + half, c - half:c + half].copy()

        # Normalise
        if self.mean is not None and self.std is not None:
            feat = (feat - self.mean[:, None, None]) / (self.std[:, None, None] + 1e-10)

        x = torch.from_numpy(feat).float()
        y = torch.from_numpy(label).float().unsqueeze(0)  # (1, P, P)
        return x, y


# Build dataset and split train/val (80/20)
# Normalisation stats are computed ONLY from training patches (no data leakage)
full_ds = LeveeDataset(
    features=features,
    labels=labels,
    patch_size=PATCH_SIZE,
    n_patches=N_PATCHES,
    mean=None,   # set after split
    std=None,
)

n_val   = int(len(full_ds) * 0.2)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)

# Compute normalisation stats from TRAIN patches only (Welford online algorithm)
# Memory-efficient: processes one patch at a time instead of concatenating all (~15 GB)
print('Computing normalisation statistics from training patches (online)...')
half = PATCH_SIZE // 2
_count = np.zeros(NUM_CHANNELS, dtype=np.float64)
_mean  = np.zeros(NUM_CHANNELS, dtype=np.float64)
_M2    = np.zeros(NUM_CHANNELS, dtype=np.float64)

for idx in train_ds.indices:
    r, c = full_ds.coords[idx]
    patch = features[:, r - half:r + half, c - half:c + half]  # (C, P, P)
    for ch in range(NUM_CHANNELS):
        vals = patch[ch].ravel().astype(np.float64)
        vals = vals[~np.isnan(vals)]
        for v in [vals]:  # process channel as a batch
            n = len(v)
            if n == 0:
                continue
            new_count = _count[ch] + n
            delta     = v.mean() - _mean[ch]
            _mean[ch] += delta * n / new_count
            # Parallel Welford: combine batch variance with running variance
            _M2[ch]   += v.var() * n + delta ** 2 * _count[ch] * n / new_count
            _count[ch] = new_count

feat_mean = _mean.astype(np.float32)
feat_std  = np.sqrt(_M2 / np.maximum(_count, 1)).astype(np.float32)
feat_std  = np.maximum(feat_std, 1e-6)  # prevent near-zero std
del _count, _mean, _M2

print(f'  feat_mean range: [{feat_mean.min():.4f}, {feat_mean.max():.4f}]')
print(f'  feat_std  range: [{feat_std.min():.4f}, {feat_std.max():.4f}]')

# Apply normalisation stats to the shared dataset
full_ds.set_norm(feat_mean, feat_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train patches: {len(train_ds):,}  |  Val patches: {len(val_ds):,}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')


---
## Section 4 — Training Loop

- Optimiser: AdamW with cosine annealing LR scheduler
- Loss: SoftClDiceLoss (Dice + clDice)
- Early stopping on validation loss
- Best model checkpoint saved automatically

In [ ]:
import json
from datetime import datetime


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        total_loss += criterion(pred, y).item()
    return total_loss / len(loader)


# Reinitialise model
model     = DLinkNet34(num_channels=NUM_CHANNELS).to(DEVICE)
criterion = SoftClDiceLoss(iters=CLDICE_ITER, alpha=CLDICE_ALPHA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

checkpoint_path = OUT_DIR / 'dlinknet_best.pt'
history = {'train_loss': [], 'val_loss': []}

best_val_loss = float('inf')
patience_counter = 0

print(f'Training D-LinkNet34 for up to {N_EPOCHS} epochs...')
print(f'  Device    : {DEVICE}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Loss      : SoftClDiceLoss (alpha={CLDICE_ALPHA})')
print()

for epoch in range(1, N_EPOCHS + 1):
    t_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    v_loss = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)

    improved = v_loss < best_val_loss
    if improved:
        best_val_loss = v_loss
        patience_counter = 0
        torch.save({
            'epoch':      epoch,
            'model_state_dict': model.state_dict(),
            'val_loss':   v_loss,
            'feat_mean':  torch.from_numpy(feat_mean),
            'feat_std':   torch.from_numpy(feat_std),
        }, checkpoint_path)
    else:
        patience_counter += 1

    mark = ' ✓' if improved else ''
    print(f'Epoch {epoch:3d}/{N_EPOCHS}  train={t_loss:.4f}  val={v_loss:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}{mark}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\nBest val loss: {best_val_loss:.4f}  →  {checkpoint_path}')


### 4.1 Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train loss')
ax.plot(history['val_loss'],   label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('clDice loss')
ax.set_title('D-LinkNet34 Training Curves')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150)
plt.show()

---
## Section 5 — Tiled Inference

Full scene processed in overlapping tiles to avoid edge artefacts.
Overlapping regions averaged in probability space before thresholding.

In [ ]:
@torch.no_grad()
def infer_tiled(
    model: nn.Module,
    features: np.ndarray,     # (C, H, W)
    mean: np.ndarray,
    std: np.ndarray,
    tile_size: int = TILE_SIZE,
    overlap: int   = TILE_OVERLAP,
    device: str    = DEVICE,
    batch_size: int = 4,
) -> np.ndarray:
    """
    Runs D-LinkNet34 inference over a full scene using overlapping tiles.
    Probability values in overlapping regions are averaged.
    Returns probability map (H, W) float32.
    """
    model.eval()
    C, H, W = features.shape

    # Pad image if smaller than tile_size
    pad_h = max(0, tile_size - H)
    pad_w = max(0, tile_size - W)
    if pad_h > 0 or pad_w > 0:
        features = np.pad(
            features,
            ((0, 0), (0, pad_h), (0, pad_w)),
            mode='reflect',
        )
        _, H, W = features.shape

    step    = tile_size - overlap

    prob_acc  = np.zeros((H, W), dtype=np.float32)
    count_acc = np.zeros((H, W), dtype=np.float32)

    # Collect all tile coordinates
    tiles = []
    r = 0
    while r < H:
        r_end = min(r + tile_size, H)
        r_start = r_end - tile_size
        c = 0
        while c < W:
            c_end   = min(c + tile_size, W)
            c_start = c_end - tile_size
            tiles.append((r_start, r_end, c_start, c_end))
            if c_end == W:
                break
            c += step
        if r_end == H:
            break
        r += step

    print(f'  Total tiles: {len(tiles)}  (tile={tile_size}, overlap={overlap})')

    # Process in batches
    for i in tqdm(range(0, len(tiles), batch_size), desc='  Inference'):
        batch_coords = tiles[i: i + batch_size]
        batch_tensors = []

        for (rs, re, cs, ce) in batch_coords:
            tile = features[:, rs:re, cs:ce].astype(np.float32)
            tile = (tile - mean[:, None, None]) / (std[:, None, None] + 1e-10)
            batch_tensors.append(torch.from_numpy(tile))

        x    = torch.stack(batch_tensors).to(device)   # (B, C, T, T)
        pred = model(x).squeeze(1).cpu().numpy()        # (B, T, T)

        for j, (rs, re, cs, ce) in enumerate(batch_coords):
            prob_acc[rs:re, cs:ce]  += pred[j]
            count_acc[rs:re, cs:ce] += 1.0

    result = prob_acc / np.maximum(count_acc, 1.0)
    # Crop back to original size if padded
    if pad_h > 0 or pad_w > 0:
        result = result[:H - pad_h, :W - pad_w]
    return result


# Load best checkpoint
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}  (val_loss={checkpoint["val_loss"]:.4f})')

print('Running tiled inference...')
prob_map = infer_tiled(
    model, features,
    mean=checkpoint['feat_mean'].numpy(),
    std=checkpoint['feat_std'].numpy(),
)

prob_path = OUT_DIR / 'probability_map.tif'
write_singleband_tiff(prob_path, prob_map, geo_ref)
print(f'Probability map saved → {prob_path}')


---
## Section 6 — Postprocessing: Skeleton → Vector

Pipeline:
1. Threshold probability map → binary mask
2. Remove small disconnected components
3. Skeletonize (Zhang-Suen) → centreline
4. Vectorize skeleton pixels → LineString GeoPackage

The vectorization connects adjacent skeleton pixels into polylines
using a graph-traversal approach.

In [ ]:
from skimage.morphology import skeletonize, remove_small_objects
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString
from scipy.ndimage import label as nd_label


def probability_to_skeleton(
    prob_map: np.ndarray,
    threshold: float = 0.5,
    min_object_size: int = 50,
) -> tuple:
    """
    Converts probability map to binary mask and computes skeleton.
    Returns (binary_mask, skeleton_mask).
    """
    binary = prob_map >= threshold
    cleaned = remove_small_objects(binary, min_size=min_object_size, connectivity=2)
    skel = skeletonize(cleaned)
    return cleaned.astype(np.uint8), skel.astype(np.uint8)


def skeleton_to_linestrings(
    skeleton: np.ndarray,
    geo_info: dict,
    min_length_px: int = 20,
) -> list:
    """
    Converts skeleton raster to list of Shapely LineStrings.
    Uses 8-connected component labelling and traces each component.
    Returns list of (LineString, length_m) tuples.
    """
    gt = geo_info['geotransform']

    def px_to_coords(r, c):
        """Pixel row/col to map coordinates (centre of pixel)."""
        x = gt[0] + (c + 0.5) * gt[1]
        y = gt[3] + (r + 0.5) * gt[5]
        return x, y

    # Label connected components (8-connectivity)
    struct = np.ones((3, 3), dtype=int)
    labeled, n_components = nd_label(skeleton, structure=struct)
    print(f'  Skeleton components: {n_components}')

    lines = []
    for comp_id in range(1, n_components + 1):
        yx = np.argwhere(labeled == comp_id)  # (N, 2)

        if len(yx) < min_length_px:
            continue

        # Sort pixels along the skeleton by proximity (greedy path)
        visited = [False] * len(yx)
        order   = [0]
        visited[0] = True

        for _ in range(len(yx) - 1):
            last = yx[order[-1]]
            dists = np.linalg.norm(yx - last, axis=1)
            dists[visited] = np.inf
            nearest = int(np.argmin(dists))
            if dists[nearest] > 3:   # gap > 3px → stop
                break
            order.append(nearest)
            visited[nearest] = True

        coords = [px_to_coords(yx[i][0], yx[i][1]) for i in order]
        if len(coords) >= 2:
            line   = LineString(coords)
            length = line.length
            lines.append((line, length))

    print(f'  LineStrings created: {len(lines)}  (min_length={min_length_px} px)')
    return lines


# ── Run postprocessing ────────────────────────────────────────────────────
THRESHOLD        = 0.5   # adjust after inspecting probability map
MIN_OBJECT_SIZE  = 50    # minimum blob size in pixels
MIN_LINE_LENGTH  = 30    # minimum skeleton length in pixels (~300m at 10m resolution)

print('Thresholding and skeletonizing...')
binary_mask, skeleton = probability_to_skeleton(
    prob_map,
    threshold=THRESHOLD,
    min_object_size=MIN_OBJECT_SIZE,
)

# Save masks for QC
write_singleband_tiff(OUT_DIR / 'binary_mask.tif',  binary_mask.astype(np.float32), geo_ref, nodata=255)
write_singleband_tiff(OUT_DIR / 'skeleton_mask.tif', skeleton.astype(np.float32),   geo_ref, nodata=255)

print('Vectorizing skeleton...')
lines = skeleton_to_linestrings(skeleton, geo_ref, min_length_px=MIN_LINE_LENGTH)

# Build GeoDataFrame
gdf_lines = gpd.GeoDataFrame(
    {
        'geometry':  [line for line, _ in lines],
        'length_m':  [length for _, length in lines],
        'source':    'DLinkNet34',
        'threshold': THRESHOLD,
    },
    crs=f'EPSG:2180',  # matches levee_detection.ipynb EPSG config
)

vector_path = OUT_DIR / 'detected_levees_dlinknet.gpkg'
gdf_lines.to_file(vector_path, driver='GPKG', engine='pyogrio')

print(f'\nVectorization complete:')
print(f'  LineStrings : {len(gdf_lines)}')
print(f'  Total length: {gdf_lines.length_m.sum() / 1000:.1f} km')
print(f'  Saved       → {vector_path}')


---
## Section 7 — Save Model Metadata

In [ ]:
metadata = {
    'created':          datetime.now().isoformat(),
    'model':            'DLinkNet34',
    'num_channels':     NUM_CHANNELS,
    'patch_size':       PATCH_SIZE,
    'best_epoch':       int(checkpoint['epoch']),
    'best_val_loss':    float(checkpoint['val_loss']),
    'cldice_alpha':     CLDICE_ALPHA,
    'cldice_iter':      CLDICE_ITER,
    'threshold':        THRESHOLD,
    'tile_size':        TILE_SIZE,
    'tile_overlap':     TILE_OVERLAP,
    'min_object_size':  MIN_OBJECT_SIZE,
    'min_line_length_px': MIN_LINE_LENGTH,
    'n_detected_lines': len(gdf_lines),
    'total_length_km':  float(gdf_lines.length_m.sum() / 1000),
    'epsg':             2180,
    'pixel_size_m':     10,
}

meta_path = OUT_DIR / 'dlinknet_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {meta_path}')

print('\n=== D-LinkNet34 pipeline complete ===')
print(f"  Checkpoint  : {checkpoint_path}")
print(f"  Prob map    : {prob_path}")
print(f"  Vector out  : {vector_path}")